# Notebook Colab pour entraîner le modèle ECG

Ce notebook est conçu pour être ouvert dans VS Code avec l’extension Colab, ou exécuté dans un runtime Google Colab. Il prépare l’environnement, vérifie le GPU, lance un entraînement court du pipeline fusion, puis montre où récupérer le modèle et les artefacts.

In [ ]:
from pathlib import Path
import sys
import subprocess

# Recherche la racine du projet à partir du dossier courant du notebook.
current = Path.cwd().resolve()
project_root = None
for candidate in [current, *current.parents]:
    if (candidate / "requirements.txt").exists() and (candidate / "config.yaml").exists():
        project_root = candidate
        break

if project_root is None:
    raise FileNotFoundError("Impossible de trouver la racine du projet. Ouvre le notebook depuis le dossier projet ou clone le dépôt avant de l'exécuter.")

print(f"Project root: {project_root}")
print(f"Python: {sys.executable}")

subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(project_root / "requirements.txt")], check=True)
print("Dépendances installées ou déjà présentes.")

In [ ]:
from pipeline_fusion.model import ECGFusionModel

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = ECGFusionModel(
    num_classes=NUM_CLASSES,
    embed_dim=256,
    pretrained=True,
    dropout=0.3,
    modality_drop_p=0.15,
).to(device)

n_params = sum(p.numel() for p in model.parameters())
print(f'Model parameters: {n_params:,}')
print(model.__class__.__name__)


In [ ]:
from sklearn.metrics import classification_report, multilabel_confusion_matrix
import numpy as np
import json

best_ckpt_path = ckpt_dir / 'best_fusion_model.pth'
if best_ckpt_path.exists():
    best_ckpt = torch.load(best_ckpt_path, map_location=device, weights_only=False)
    model.load_state_dict(best_ckpt['model_state_dict'], strict=False)
    trainer.model = model.to(device)

    test_metrics, preds, labels = trainer.validate(test_loader)
    print('Test metrics:', json.dumps(test_metrics, indent=2, default=str))
    print(classification_report(labels, preds, target_names=CLASS_NAMES, zero_division=0))
    print('Confusion matrices:')
    for idx, matrix in enumerate(multilabel_confusion_matrix(labels, preds)):
        print(CLASS_NAMES[idx], matrix.tolist())
else:
    print(f'Checkpoint not found: {best_ckpt_path}')

print(f'Best checkpoint: {best_ckpt_path}')
print(f'Run manifest: {ckpt_dir / "run_manifest.json"}')
print(f'Training metrics: {ckpt_dir / "training_metrics.jsonl"}')

## 6. Évaluation et sauvegarde du modèle

Après l’entraînement, cette section recharge le meilleur checkpoint, évalue le modèle sur le test set et rappelle où se trouvent les artefacts enregistrés.

In [ ]:
from common.losses import MultiLabelLogitAdjustedBCE
from pipeline_fusion.trainer import FusionTrainer

trainer_config = {
    'learning_rate': 3e-4,
    'weight_decay': 1e-4,
    'patience': 10,
}

trainer = FusionTrainer(model, trainer_config, device, logger)
trainer.setup_loss(class_counts, loss_type='logit_adj', tau=1.0)

EPOCHS = 5
BATCH_SIZE = 4
MAX_BATCHES = None

# Recrée les loaders si tu veux changer la taille de batch.
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

ckpt_dir = project_root / 'models' / 'checkpoints'
ckpt_dir.mkdir(parents=True, exist_ok=True)

history = trainer.train_loop(
    train_loader,
    val_loader,
    epochs=EPOCHS,
    ckpt_dir=str(ckpt_dir),
    start_epoch=1,
    max_batches=MAX_BATCHES,
)

print('Training history keys:', history.keys())

## 5. Entraînement du modèle

Lance ici l’entraînement du modèle fusion. Sur Colab, augmente `EPOCHS` si le GPU est disponible et que tu veux un run plus long.

## 4. Définition du modèle

Cette section instancie le modèle fusion actuel. L’architecture combine le backbone signal, le backbone image, la co-attention et les métadonnées cliniques.

In [ ]:
from common.utils import set_seed, load_config, CLASS_NAMES, NUM_CLASSES
from pipeline_fusion.dataset import load_fusion_dataset, ECGDualDataset
from torch.utils.data import DataLoader

set_seed(42)
config = load_config(str(project_root / 'config.yaml'))

csv_path = project_root / config['paths']['ptbxl_signals'] / 'ptbxl_database.csv'
signals_dir = project_root / config['paths']['ptbxl_signals']
images_dir = project_root / config['paths']['ptbxl_images']

logger_name = 'colab_notebook'
import logging
logger = logging.getLogger(logger_name)
if not logger.handlers:
    logging.basicConfig(level=logging.INFO)

full_df = load_fusion_dataset(str(csv_path), str(signals_dir), str(images_dir), logger)
train_df = full_df[full_df['strat_fold'].isin([1, 2, 3, 4, 5, 6, 7, 8])].reset_index(drop=True)
val_df = full_df[full_df['strat_fold'] == 9].reset_index(drop=True)
test_df = full_df[full_df['strat_fold'] == 10].reset_index(drop=True)

train_ds = ECGDualDataset(train_df, augment=True)
val_ds = ECGDualDataset(val_df, augment=False)
test_ds = ECGDualDataset(test_df, augment=False)

train_loader = DataLoader(train_ds, batch_size=2, shuffle=True, num_workers=0)
val_loader = DataLoader(val_ds, batch_size=2, shuffle=False, num_workers=0)
test_loader = DataLoader(test_ds, batch_size=2, shuffle=False, num_workers=0)

class_counts = train_df['target'].tolist()
class_counts = __import__('numpy').stack(class_counts).sum(axis=0)
print('Classes:', CLASS_NAMES)
print('Train size:', len(train_ds), 'Val size:', len(val_ds), 'Test size:', len(test_ds))
print('Class counts:', class_counts.tolist())

## 3. Préparation des données d’entraînement

La cellule suivante charge les métadonnées PTB-XL, crée les splits et prépare les datasets fusion. Elle peut servir sur Colab si les fichiers du projet et le dataset sont accessibles.

In [ ]:
import os
import json
from pathlib import Path

import torch

print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("GPU non disponible, exécution CPU.")

os.chdir(project_root)
print(f"Working directory: {Path.cwd()}")
print(f"config.yaml exists: {(project_root / 'config.yaml').exists()}")
print(f"dataset csv exists: {(project_root / 'PTB-XL ECG dataset' / 'ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.1' / 'ptbxl_database.csv').exists()}")

## 2. Connexion à l’environnement Colab

Cette section vérifie le runtime, l’accès au GPU et le chemin du projet. Si tu es sur Colab distant, assure-toi que le dépôt ou les données sont bien accessibles dans l’environnement.

## 1. Installation et configuration de l’extension Google Colab dans VS Code

Ouvre ce notebook avec l’extension Colab installée dans VS Code, puis vérifie que l’environnement Python est bien sélectionné. La cellule suivante installe les dépendances du projet si elles ne sont pas déjà présentes.